In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from package.RankAMIP.logistic import run_logistic_regression
from package.RankAMIP.data_script import make_BT_design_matrix
from package.RankAMIP.logistic import LogisticAMIP
from package.RankAMIP.logistic import find_closest_matchups
from package.RankAMIP.logistic import isRankingRobust
from package.RankAMIP.data_script import *

### How Robust is the Multi-turn Benchmark to Data-Dropping?

The MT-Bench (Multi-Turn Benchmark) is a curated set of 80 multi-turn dialogue prompts designed to evaluate the conversational and instruction-following capabilities of large language models (LLMs). Each prompt simulates realistic, multi-turn interactions that test a model's ability to maintain context, reason logically, and provide coherent responses across various domains, including general knowledge, reasoning, programming, and open-ended tasks.

### Load Data

In [2]:
# Import datasets from 
# https://huggingface.co/datasets/lmsys/mt_bench_human_judgments
from datasets import load_dataset
ds = load_dataset("lmsys/mt_bench_human_judgments")

/Users/JennyH/Library/Python/3.8/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# inspect the available splits
print(ds)  
# MT-bench has both human and a gpt4-judge data.
gpt4_pair = ds["gpt4_pair"] 
human = ds["human"] 
# look at the first example
print(gpt4_pair[0])

DatasetDict({
    gpt4_pair: Dataset({
        features: ['question_id', 'model_a', 'model_b', 'winner', 'judge', 'conversation_a', 'conversation_b', 'turn'],
        num_rows: 2400
    })
    human: Dataset({
        features: ['question_id', 'model_a', 'model_b', 'winner', 'judge', 'conversation_a', 'conversation_b', 'turn'],
        num_rows: 3355
    })
})
{'question_id': 81, 'model_a': 'alpaca-13b', 'model_b': 'claude-v1', 'winner': 'model_b', 'judge': 'gpt4_pair', 'conversation_a': [{'content': 'Compose an engaging travel blog post about a recent trip to Hawaii, highlighting cultural experiences and must-see attractions.', 'role': 'user'}, {'content': 'I recently had the pleasure of visiting Hawaii and it quickly became one of my favorite places. From the stunning beaches to the lush mountains, this place has it all. The people are incredibly friendly and the culture is alive and well. One of the highlights of my trip was visiting the Polynesian Cultural Center. Here, I was able 

In [4]:
df = human.to_pandas()
df.shape

(3355, 8)

In [5]:
# create a column winner_model_a, which is 1 if model_a is preferred, 0 if model_b is preferred
df['winner_model_a'] = df['winner'].apply(lambda x: 1 if x == 'model_a' else 0)
# create a column called winner_tie that is 1 if the winner is 'tie', else 0
df['winner_tie'] = df['winner'].apply(lambda x: 1 if x == 'tie' else 0)
df.head()

,question_id,model_a,model_b,winner,judge,conversation_a,conversation_b,turn,winner_model_a,winner_tie
0,81,alpaca-13b,gpt-3.5-turbo,model_b,author_2,[{'content': 'Compose an engaging travel blog ...,[{'content': 'Compose an engaging travel blog ...,1,0,0
1,81,alpaca-13b,gpt-3.5-turbo,model_b,author_2,[{'content': 'Compose an engaging travel blog ...,[{'content': 'Compose an engaging travel blog ...,2,0,0
2,81,alpaca-13b,gpt-3.5-turbo,model_b,expert_17,[{'content': 'Compose an engaging travel blog ...,[{'content': 'Compose an engaging travel blog ...,1,0,0
3,81,alpaca-13b,gpt-3.5-turbo,model_b,expert_17,[{'content': 'Compose an engaging travel blog ...,[{'content': 'Compose an engaging travel blog ...,2,0,0
4,81,alpaca-13b,vicuna-13b-v1.2,model_b,expert_0,[{'content': 'Compose an engaging travel blog ...,[{'content': 'Compose an engaging travel blog ...,1,0,0


In [6]:
ties = df[df['winner_tie'] == 1]
print(f"Number of ties: {len(ties)}")
# proportion of ties.
print(f"Proportion of ties: {len(ties) / len(df):.2%}")
# note, the proportion of ties is 9.17% for the LLM-as-judge data and 23.25% for the human-as-judge data.

Number of ties: 780
Proportion of ties: 23.25%


In [7]:
rawBT = df[['model_a', 'model_b', 'winner_model_a', 'winner_tie']]
rawBT.head()
rawBT.shape
# rawBT_noTies.head() # (2575, 3)
# rawBT_noTies.shape

(3355, 4)

In [8]:
# how to get the unique names in both columns
model_a_names = df['model_a'].unique()
model_b_names = df['model_b'].unique()
# combine the two arrays and get the unique names
model_names = np.unique(np.concatenate((model_a_names, model_b_names)))
# print the number of unique model names
print(f"Number of unique model names: {len(model_names)}")

Number of unique model names: 6


In [9]:
for model in model_names:
    filtered = df[
        (df['model_a'] == model) | 
        (df['model_b'] == model)
    ]
    print(f"{model}: {filtered.shape[0]}")

alpaca-13b: 1018
claude-v1: 996
gpt-3.5-turbo: 1478
gpt-4: 1029
llama-13b: 1083
vicuna-13b-v1.2: 1106


In [11]:
# make the BT design matrix.
X, y, player_to_id = make_BT_design_matrix(rawBT, weight_tie = True)
X.shape, y.shape

((6710, 5), (6710,))

#### Run Top-k Robustness Check.

In [12]:
ks = [1, 3, 5]
results = {}
for k in ks:
    alphaN = 1
    chatbotA = -1
    while chatbotA == -1:
        chatbotA, chatbotB, chatbotOriginalBetaDiff, chatNewBetaDiff, chatIndices = isRankingRobust(k, alphaN, X, y, weighted = True)
        results[(k, alphaN)] = (chatbotA, chatbotB, chatbotOriginalBetaDiff, chatNewBetaDiff, chatIndices)
        alphaN += 1

In [13]:
# find the (k, alpha N) pairs that are non-robust.
results_nonrobust = {k: v for k, v in results.items() if v[0] != -1}
results_nonrobust

{(1, 92): (2,
  0,
  0.37829896502671123,
  -0.004105539718147977,
  array([ 137, 2399, 1298, 1884, 2398,  139, 1153,  850,  391, 1111, 3181,
           91,  648, 2612,  803,  802,  804,  801,  800,  348,  744,   41,
         2726,  349, 2668,  608,  607, 1450,  799, 2909, 1409, 2912, 2725,
          748, 2492, 1537,  160, 1536, 2911, 1534,  925, 1535, 2333, 2161,
          570, 1830,  346, 2334,  745, 1408, 1191, 2332, 3055,  101,  222,
         2883, 3274,  221, 2837,  219,  667,  178, 3021, 3022, 1902, 2552,
         2551, 2341,  863, 1124, 1903, 2624, 2626, 2627, 1634,  898, 1744,
         2510, 1745,  220, 3275,  666, 1162,  246, 1214, 1294, 1165,   64,
          247, 1556,   65, 3278])),
 (3, 209): (0,
  3,
  0.6997158861364732,
  -0.0020451894492634626,
  array([1398,  331,   24, 1183, 2606, 3109, 1221, 1345, 1268, 3203, 2110,
          597, 1918,   78, 1275, 2090, 2089,  448, 3112,  449, 3297, 3299,
         3113, 2807, 2808, 2810, 1921,  294,  295, 2091,  701, 1274, 1717,
    

In [ ]:
# # save results as a .pkl file.
# import pickle
# with open('results/MTBenchHumanNonrobustWtd.pkl', 'wb') as f:
#     pickle.dump(results_nonrobust, f)

In [15]:
for model in model_names:
    filtered = df[
        (df['model_a'] == model) | 
        (df['model_b'] == model)
    ]
    print(f"{model}: {filtered.shape[0]}")

alpaca-13b: 1018
claude-v1: 996
gpt-3.5-turbo: 1478
gpt-4: 1029
llama-13b: 1083
vicuna-13b-v1.2: 1106


In [16]:
from package.RankAMIP.plot_util import *
rankings = return_rankings_list(X, y, results, 1, 92, player_to_id)

In [17]:
# plot the rankings on the original arena
filename_to_save = 'fig/top6_mtb_human.png'
plot_title = 'Model Rankings in MT-Bench'
plot_bt_scores(X, y, rankings, alphaN, 6, plot_title, filename_to_save)